# 🏭 Notebook 00 — Environment Setup
## Unsupervised Industrial Defect Detection | MVTec AD Benchmark

**Project:** PatchCore vs Convolutional Autoencoder  
**Purpose:** Initialize environment, folder structure, config, and reproducibility settings  
**Platform:** Google Colab Free Tier (T4 GPU) + Google Drive  

> ⚠️ **Run this notebook ONCE at the start of every new Colab session.**  
> All subsequent notebooks depend on paths and configs defined here.

---

### STEP 0 — Install Dependencies

In [1]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — Dependency management for Google Colab
#
# Strategy:
#   - Colab pre-installs: torch, torchvision, numpy, matplotlib,
#     seaborn, tqdm, opencv, scikit-learn → DO NOT reinstall
#   - Missing from Colab: faiss-cpu, pytorch-msssim → install these
#   - Verify minimum version requirements after installation
#
# Why not pin torch version:
#   Colab ships with a CUDA-matched torch build. Overriding it breaks
#   the CUDA runtime. We accept whatever version Colab provides
#   as long as it meets the minimum (>=2.0.0).
# ─────────────────────────────────────────────────────────────────

import subprocess, sys, importlib


def pip_install(package: str, quiet: bool = True):
    """Install a package via pip. Raises on failure."""
    flags = ["--quiet"] if quiet else []
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install"] + flags + [package],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"pip install failed for '{package}':\n{result.stderr.strip()}"
        )


def is_importable(module_name: str) -> bool:
    """Check if a module can be imported without actually importing it."""
    return importlib.util.find_spec(module_name) is not None


# ── Packages to install only if NOT already present ───────────────
# These are absent from Colab's default environment.
INSTALL_IF_MISSING = [
    ("faiss",          "faiss-cpu"),       # FAISS: k-NN for PatchCore memory bank
    ("pytorch_msssim", "pytorch-msssim"),  # SSIM loss for Conv-AE training
    ("psutil",         "psutil"),           # System memory monitoring
]

# ── Packages expected to already exist in Colab ───────────────────
# We verify these are importable but do NOT reinstall them.
COLAB_PREINSTALLED = [
    "torch", "torchvision", "numpy", "sklearn",
    "cv2", "matplotlib", "seaborn", "tqdm",
    "json", "pathlib", "gc",
]

# ── Step A: Install missing packages ─────────────────────────────
print("📦 Installing missing packages...")
for module_name, pip_name in INSTALL_IF_MISSING:
    if is_importable(module_name):
        print(f"  ⏭️  {pip_name:<22} already installed — skipped")
    else:
        try:
            pip_install(pip_name)
            print(f"  ✅ {pip_name:<22} installed")
        except RuntimeError as e:
            print(f"  ❌ {pip_name:<22} FAILED")
            raise

# ── Step B: Verify Colab pre-installed packages are importable ────
print("\n🔍 Verifying Colab pre-installed packages...")
missing_preinstalled = []
for module_name in COLAB_PREINSTALLED:
    if is_importable(module_name):
        print(f"  ✅ {module_name}")
    else:
        print(f"  ❌ {module_name} — NOT FOUND (unexpected)")
        missing_preinstalled.append(module_name)

if missing_preinstalled:
    raise EnvironmentError(
        f"\n❌ Expected Colab packages not found: {missing_preinstalled}\n"
        f"This is unusual. Try Runtime > Disconnect and delete runtime, then reconnect."
    )

# ── Step C: Minimum version check for torch ───────────────────────
import torch
torch_version = tuple(int(x) for x in torch.__version__.split(".")[:2])
if torch_version < (2, 0):
    raise EnvironmentError(
        f"\n❌ torch >= 2.0.0 required. Found: {torch.__version__}\n"
        f"Please use a Colab runtime with a newer PyTorch version."
    )

print(f"\n  torch version : {torch.__version__} (>= 2.0.0 ✅)")
print("\n✅ All dependencies ready.")

📦 Installing missing packages...
  ✅ faiss-cpu              installed
  ✅ pytorch-msssim         installed
  ⏭️  psutil                 already installed — skipped

🔍 Verifying Colab pre-installed packages...
  ✅ torch
  ✅ torchvision
  ✅ numpy
  ✅ sklearn
  ✅ cv2
  ✅ matplotlib
  ✅ seaborn
  ✅ tqdm
  ✅ json
  ✅ pathlib
  ✅ gc

  torch version : 2.10.0+cpu (>= 2.0.0 ✅)

✅ All dependencies ready.


In [2]:
# ─────────────────────────────────────────────────────────────────
# STEP 0b — Suppress non-critical warnings for cleaner output
# ─────────────────────────────────────────────────────────────────

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Suppress torchvision pretrained weight deprecation warnings
import os
os.environ["PYTHONWARNINGS"] = "ignore"

print("✅ Warnings suppressed.")

✅ Warnings suppressed.


---
### STEP 1 — Mount Google Drive

In [3]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — Mount Google Drive
# Drive is used ONLY for persistent storage of final artifacts.
# All heavy computation runs on local /content/ SSD.
# ─────────────────────────────────────────────────────────────────

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# Verify mount succeeded
DRIVE_MYDRIVE = "/content/drive/MyDrive"

if not os.path.exists(DRIVE_MYDRIVE):
    raise EnvironmentError(
        f"❌ Google Drive mount FAILED. "
        f"Expected path not found: {DRIVE_MYDRIVE}\n"
        f"Please re-run this cell and authorize Drive access."
    )

print(f"✅ Google Drive mounted successfully at: {DRIVE_MYDRIVE}")

Mounted at /content/drive
✅ Google Drive mounted successfully at: /content/drive/MyDrive


---
### STEP 2 — Define Project Root (Single Source of Truth)

In [4]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — Project root definitions
#
# ⚠️  SINGLE SOURCE OF TRUTH — Do NOT hardcode these paths
#     in any other notebook. Always load from config.json instead.
# ─────────────────────────────────────────────────────────────────

# --- Project identity ---
PROJECT_NAME = "industrial_anomaly_project"

# --- Google Drive root (persistent storage) ---
DRIVE_ROOT = f"/content/drive/MyDrive/{PROJECT_NAME}"

# --- Local Colab SSD root (fast I/O workspace) ---
LOCAL_ROOT = "/content/project_workspace"

print(f"PROJECT_NAME : {PROJECT_NAME}")
print(f"DRIVE_ROOT   : {DRIVE_ROOT}")
print(f"LOCAL_ROOT   : {LOCAL_ROOT}")

PROJECT_NAME : industrial_anomaly_project
DRIVE_ROOT   : /content/drive/MyDrive/industrial_anomaly_project
LOCAL_ROOT   : /content/project_workspace


---
### STEP 3 — Define Full Directory Structure

In [5]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Full directory structure definition
#
# Google Drive paths  → persistent across sessions (final artifacts only)
# Local /content/ paths → ephemeral, fast SSD (all heavy computation)
# ─────────────────────────────────────────────────────────────────

# ── Google Drive: Persistent Storage ─────────────────────────────
DATASET_DRIVE  = f"{DRIVE_ROOT}/data"        # raw downloaded datasets
MODEL_DRIVE    = f"{DRIVE_ROOT}/models"      # .pt checkpoints + .npy memory banks
RESULTS_DRIVE  = f"{DRIVE_ROOT}/results"     # experiment_log.csv + figures
LOGS_DRIVE     = f"{DRIVE_ROOT}/logs"        # training loss logs
CONFIG_DRIVE   = f"{DRIVE_ROOT}/configs"     # config.json + thresholds.json

# ── Local /content/: Fast SSD Workspace ──────────────────────────
LOCAL_DATA     = f"{LOCAL_ROOT}/data"        # dataset copied from Drive for fast I/O
LOCAL_TEMP     = f"{LOCAL_ROOT}/temp"        # intermediate files (patch features, etc.)
LOCAL_MODELS   = f"{LOCAL_ROOT}/models"      # local model cache during training
LOCAL_CACHE    = f"{LOCAL_ROOT}/cache"       # miscellaneous cache (FAISS indices, etc.)

print("📂 Directory structure defined:")
print()
print("── Google Drive (Persistent) ──")
for name, path in [
    ("DATASET_DRIVE",  DATASET_DRIVE),
    ("MODEL_DRIVE",    MODEL_DRIVE),
    ("RESULTS_DRIVE",  RESULTS_DRIVE),
    ("LOGS_DRIVE",     LOGS_DRIVE),
    ("CONFIG_DRIVE",   CONFIG_DRIVE),
]:
    print(f"  {name:<18} → {path}")

print()
print("── Local /content/ (Fast SSD) ──")
for name, path in [
    ("LOCAL_DATA",     LOCAL_DATA),
    ("LOCAL_TEMP",     LOCAL_TEMP),
    ("LOCAL_MODELS",   LOCAL_MODELS),
    ("LOCAL_CACHE",    LOCAL_CACHE),
]:
    print(f"  {name:<18} → {path}")

📂 Directory structure defined:

── Google Drive (Persistent) ──
  DATASET_DRIVE      → /content/drive/MyDrive/industrial_anomaly_project/data
  MODEL_DRIVE        → /content/drive/MyDrive/industrial_anomaly_project/models
  RESULTS_DRIVE      → /content/drive/MyDrive/industrial_anomaly_project/results
  LOGS_DRIVE         → /content/drive/MyDrive/industrial_anomaly_project/logs
  CONFIG_DRIVE       → /content/drive/MyDrive/industrial_anomaly_project/configs

── Local /content/ (Fast SSD) ──
  LOCAL_DATA         → /content/project_workspace/data
  LOCAL_TEMP         → /content/project_workspace/temp
  LOCAL_MODELS       → /content/project_workspace/models
  LOCAL_CACHE        → /content/project_workspace/cache


---
### STEP 4 — Auto-Create All Directories

In [6]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — Auto-create all directories
# Uses exist_ok=True so this is safe to re-run on session restart.
# ─────────────────────────────────────────────────────────────────

ALL_DIRS = [
    # Drive directories
    DRIVE_ROOT,
    DATASET_DRIVE,
    MODEL_DRIVE,
    RESULTS_DRIVE,
    LOGS_DRIVE,
    CONFIG_DRIVE,
    # Local directories
    LOCAL_ROOT,
    LOCAL_DATA,
    LOCAL_TEMP,
    LOCAL_MODELS,
    LOCAL_CACHE,
]

print("🏗️  Creating directories...")
failed = []

for d in ALL_DIRS:
    os.makedirs(d, exist_ok=True)
    exists = os.path.exists(d)
    status = "✅ OK" if exists else "❌ FAILED"
    print(f"  {status}  {d}")
    if not exists:
        failed.append(d)

if failed:
    raise Exception(
        f"\n❌ Directory creation FAILED for {len(failed)} path(s):\n"
        + "\n".join(f"  - {d}" for d in failed)
    )

print(f"\n✅ All {len(ALL_DIRS)} directories created successfully.")

🏗️  Creating directories...
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project/data
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project/models
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project/results
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project/logs
  ✅ OK  /content/drive/MyDrive/industrial_anomaly_project/configs
  ✅ OK  /content/project_workspace
  ✅ OK  /content/project_workspace/data
  ✅ OK  /content/project_workspace/temp
  ✅ OK  /content/project_workspace/models
  ✅ OK  /content/project_workspace/cache

✅ All 11 directories created successfully.


---
### STEP 5 — Set Random Seed (Reproducibility)

In [7]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — Global random seed for full reproducibility
# Applies to: Python random, NumPy, PyTorch CPU, PyTorch CUDA
# ─────────────────────────────────────────────────────────────────

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Enforce deterministic CUDA operations (slight performance cost, required for reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"✅ Random seed set to {SEED} across: random | numpy | torch | cuda")
print(f"   cudnn.deterministic = True | cudnn.benchmark = False")

✅ Random seed set to 42 across: random | numpy | torch | cuda
   cudnn.deterministic = True | cudnn.benchmark = False


---
### STEP 6 — GPU Check

In [8]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — GPU availability check
# Falls back to CPU gracefully (with warning) — does not raise error.
# PatchCore feature extraction requires GPU for reasonable speed.
# Conv-AE training will be very slow on CPU (~10x slower than T4).
# ─────────────────────────────────────────────────────────────────

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name  = torch.cuda.get_device_name(0)
    gpu_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected:")
    print(f"   Device     : {gpu_name}")
    print(f"   Total VRAM : {gpu_total:.2f} GB")
    print(f"   CUDA ver.  : {torch.version.cuda}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  WARNING: No GPU detected. Falling back to CPU.")
    print("   PatchCore feature extraction will be significantly slower.")
    print("   Conv-AE training is not recommended without GPU.")
    print("   → Go to Runtime > Change runtime type > GPU (T4)")

print(f"\n   Active device: {DEVICE}")

⚠️  WARNING: No GPU detected. Falling back to CPU.
   PatchCore feature extraction will be significantly slower.
   Conv-AE training is not recommended without GPU.
   → Go to Runtime > Change runtime type > GPU (T4)

   Active device: cpu


---
### STEP 7 — Memory Utilities

In [9]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — Memory utility functions
#
# clear_memory()     : Call at end of every major pipeline stage
# print_system_info(): Call to inspect RAM/VRAM usage at any point
# ─────────────────────────────────────────────────────────────────

import gc
import psutil


def clear_memory():
    """Release Python garbage and clear CUDA memory cache.
    Call this at the end of every major computation block.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory cleared (GC + CUDA cache).")


def print_system_info():
    """Print current RAM and GPU VRAM usage."""
    vm = psutil.virtual_memory()
    ram_used  = vm.used  / (1024 ** 3)
    ram_total = vm.total / (1024 ** 3)
    ram_pct   = vm.percent

    print("─" * 45)
    print("📊 System Info")
    print(f"   RAM  : {ram_used:.2f} GB / {ram_total:.2f} GB  ({ram_pct:.1f}% used)")

    if torch.cuda.is_available():
        vram_used  = torch.cuda.memory_allocated(0)  / (1024 ** 3)
        vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        vram_pct   = vram_used / vram_total * 100
        print(f"   VRAM : {vram_used:.2f} GB / {vram_total:.2f} GB  ({vram_pct:.1f}% used)")
        print(f"   GPU  : {torch.cuda.get_device_name(0)}")
    else:
        print("   VRAM : N/A (no GPU)")
    print("─" * 45)


# Smoke test: run immediately to confirm functions work
print_system_info()

─────────────────────────────────────────────
📊 System Info
   RAM  : 0.86 GB / 12.67 GB  (9.3% used)
   VRAM : N/A (no GPU)
─────────────────────────────────────────────


---
### STEP 8 — Workflow Rules

> These rules are enforced across ALL notebooks in this project.

| # | Rule | Detail |
|---|------|--------|
| **RULE 1** | All heavy computation runs on `/content/` | Local SSD is ~100x faster than Drive I/O |
| **RULE 2** | Google Drive is for **final artifacts only** | Memory banks (.npy), checkpoints (.pt), CSVs, figures |
| **RULE 3** | Never train directly from Drive paths | Always copy dataset to `LOCAL_DATA` first |
| **RULE 4** | Call `clear_memory()` after every major stage | Prevents RAM/VRAM accumulation across experiments |

---
### STEP 9 — Write Config File to Drive

In [10]:
# ─────────────────────────────────────────────────────────────────
# STEP 9 — Persist project configuration to Google Drive
#
# config.json is the single source of truth for all path constants.
# Load it at the start of every subsequent notebook instead of
# re-defining paths manually.
#
# Usage in other notebooks:
#   import json
#   with open("/content/drive/MyDrive/industrial_anomaly_project/configs/config.json") as f:
#       cfg = json.load(f)
# ─────────────────────────────────────────────────────────────────

import json

config = {
    # ── Identity ──────────────────────────────────────────────────
    "PROJECT_NAME": PROJECT_NAME,
    "SEED":         SEED,
    "DEVICE":       str(DEVICE),

    # ── Google Drive paths (persistent) ───────────────────────────
    "DRIVE_ROOT":     DRIVE_ROOT,
    "DATASET_DRIVE":  DATASET_DRIVE,
    "MODEL_DRIVE":    MODEL_DRIVE,
    "RESULTS_DRIVE":  RESULTS_DRIVE,
    "LOGS_DRIVE":     LOGS_DRIVE,
    "CONFIG_DRIVE":   CONFIG_DRIVE,

    # ── Local /content/ paths (ephemeral) ─────────────────────────
    "LOCAL_ROOT":     LOCAL_ROOT,
    "LOCAL_DATA":     LOCAL_DATA,
    "LOCAL_TEMP":     LOCAL_TEMP,
    "LOCAL_MODELS":   LOCAL_MODELS,
    "LOCAL_CACHE":    LOCAL_CACHE,

    # ── Dataset config ────────────────────────────────────────────
    "CATEGORIES":     ["leather", "tile", "metal_nut"],
    "IMAGE_SIZE":     256,
    "IMAGENET_MEAN":  [0.485, 0.456, 0.406],
    "IMAGENET_STD":   [0.229, 0.224, 0.225],

    # ── PatchCore config ──────────────────────────────────────────
    "PATCHCORE_BACKBONE":       "wide_resnet50_2",
    "PATCHCORE_LAYERS":         ["layer2", "layer3"],
    "PATCHCORE_CORESET_RATIO":  0.10,
    "PATCHCORE_KNN_K":          9,

    # ── Conv-AE config ────────────────────────────────────────────
    "AE_EPOCHS":        50,
    "AE_BATCH_SIZE":    16,
    "AE_LR":            1e-3,
    "AE_WEIGHT_DECAY":  1e-5,
    "AE_LOSS_ALPHA":    0.8,

    # ── Evaluation config ─────────────────────────────────────────
    "VAL_NORMAL_COUNT":     10,
    "VAL_DEFECTIVE_COUNT":  10,
    "THRESHOLD_STEPS":      1000,
}

CONFIG_PATH = f"{CONFIG_DRIVE}/config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=4)

# Verify file was written
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"❌ Config file not written to: {CONFIG_PATH}")

file_size_kb = os.path.getsize(CONFIG_PATH) / 1024
print(f"✅ Config saved to: {CONFIG_PATH}")
print(f"   File size: {file_size_kb:.2f} KB")
print(f"   Keys: {list(config.keys())}")

✅ Config saved to: /content/drive/MyDrive/industrial_anomaly_project/configs/config.json
   File size: 1.43 KB
   Keys: ['PROJECT_NAME', 'SEED', 'DEVICE', 'DRIVE_ROOT', 'DATASET_DRIVE', 'MODEL_DRIVE', 'RESULTS_DRIVE', 'LOGS_DRIVE', 'CONFIG_DRIVE', 'LOCAL_ROOT', 'LOCAL_DATA', 'LOCAL_TEMP', 'LOCAL_MODELS', 'LOCAL_CACHE', 'CATEGORIES', 'IMAGE_SIZE', 'IMAGENET_MEAN', 'IMAGENET_STD', 'PATCHCORE_BACKBONE', 'PATCHCORE_LAYERS', 'PATCHCORE_CORESET_RATIO', 'PATCHCORE_KNN_K', 'AE_EPOCHS', 'AE_BATCH_SIZE', 'AE_LR', 'AE_WEIGHT_DECAY', 'AE_LOSS_ALPHA', 'VAL_NORMAL_COUNT', 'VAL_DEFECTIVE_COUNT', 'THRESHOLD_STEPS']


In [11]:
# ─────────────────────────────────────────────────────────────────
# STEP 9b — Verify config roundtrip (write → read → validate)
# ─────────────────────────────────────────────────────────────────

with open(CONFIG_PATH, "r") as f:
    loaded_config = json.load(f)

# Check critical keys survive roundtrip
CRITICAL_KEYS = [
    "PROJECT_NAME", "SEED", "DRIVE_ROOT", "LOCAL_ROOT",
    "CATEGORIES", "IMAGE_SIZE",
    "PATCHCORE_CORESET_RATIO", "AE_EPOCHS"
]

print("🔍 Config roundtrip validation:")
all_ok = True
for key in CRITICAL_KEYS:
    original = config[key]
    loaded   = loaded_config.get(key)
    ok       = original == loaded
    print(f"   {'✅' if ok else '❌'} {key}: {loaded}")
    if not ok:
        all_ok = False

if not all_ok:
    raise ValueError("❌ Config roundtrip validation FAILED — values mismatch.")

print("\n✅ Config roundtrip validation passed.")

🔍 Config roundtrip validation:
   ✅ PROJECT_NAME: industrial_anomaly_project
   ✅ SEED: 42
   ✅ DRIVE_ROOT: /content/drive/MyDrive/industrial_anomaly_project
   ✅ LOCAL_ROOT: /content/project_workspace
   ✅ CATEGORIES: ['leather', 'tile', 'metal_nut']
   ✅ IMAGE_SIZE: 256
   ✅ PATCHCORE_CORESET_RATIO: 0.1
   ✅ AE_EPOCHS: 50

✅ Config roundtrip validation passed.


---
### STEP 10 — Final Environment Verification

In [12]:
# ─────────────────────────────────────────────────────────────────
# STEP 10 — Final import verification
# Confirms all installed packages are importable before any pipeline runs.
# ─────────────────────────────────────────────────────────────────

print("🔍 Verifying all imports...\n")

import_checks = [
    ("torch",             "torch"),
    ("torchvision",       "torchvision"),
    ("faiss",             "faiss"),
    ("sklearn",           "sklearn"),
    ("cv2",               "cv2"),
    ("matplotlib",        "matplotlib"),
    ("seaborn",           "seaborn"),
    ("tqdm",              "tqdm"),
    ("pytorch_msssim",    "pytorch_msssim"),
    ("psutil",            "psutil"),
    ("numpy",             "numpy"),
    ("json",              "json"),
    ("pathlib",           "pathlib"),
]

failed_imports = []
for display_name, module_name in import_checks:
    try:
        mod = __import__(module_name)
        version = getattr(mod, "__version__", "n/a")
        print(f"  ✅ {display_name:<20} v{version}")
    except ImportError as e:
        print(f"  ❌ {display_name:<20} IMPORT FAILED: {e}")
        failed_imports.append(display_name)

if failed_imports:
    raise ImportError(
        f"\n❌ Import verification FAILED for: {failed_imports}\n"
        f"Re-run STEP 0 to reinstall dependencies."
    )

print(f"\n✅ All {len(import_checks)} imports verified.")

🔍 Verifying all imports...

  ✅ torch                v2.10.0+cpu
  ✅ torchvision          v0.25.0+cpu
  ✅ faiss                v1.13.2
  ✅ sklearn              v1.6.1
  ✅ cv2                  v4.13.0
  ✅ matplotlib           v3.10.0
  ✅ seaborn              v0.13.2
  ✅ tqdm                 v4.67.3
  ✅ pytorch_msssim       vn/a
  ✅ psutil               v5.9.5
  ✅ numpy                v2.0.2
  ✅ json                 v2.0.9
  ✅ pathlib              vn/a

✅ All 13 imports verified.


In [13]:
# ─────────────────────────────────────────────────────────────────
# STEP 10b — Print final system snapshot
# ─────────────────────────────────────────────────────────────────

print_system_info()
clear_memory()

print()
print("═" * 55)
print("✅ Notebook 00 COMPLETE — Environment Ready")
print("═" * 55)
print()
print("📌 Next step: Run 01_eda.ipynb")
print()
print("Summary:")
print(f"  Project     : {PROJECT_NAME}")
print(f"  Drive root  : {DRIVE_ROOT}")
print(f"  Local root  : {LOCAL_ROOT}")
print(f"  Device      : {DEVICE}")
print(f"  Seed        : {SEED}")
print(f"  Config      : {CONFIG_PATH}")

─────────────────────────────────────────────
📊 System Info
   RAM  : 1.18 GB / 12.67 GB  (11.8% used)
   VRAM : N/A (no GPU)
─────────────────────────────────────────────
🧹 Memory cleared (GC + CUDA cache).

═══════════════════════════════════════════════════════
✅ Notebook 00 COMPLETE — Environment Ready
═══════════════════════════════════════════════════════

📌 Next step: Run 01_eda.ipynb

Summary:
  Project     : industrial_anomaly_project
  Drive root  : /content/drive/MyDrive/industrial_anomaly_project
  Local root  : /content/project_workspace
  Device      : cpu
  Seed        : 42
  Config      : /content/drive/MyDrive/industrial_anomaly_project/configs/config.json
